# AmanLeb — Retrieval-Augmented Generation (RAG)

This notebook develops the trusted-evidence retrieval and explanation stage of AmanLeb.

The multilingual Transformer classifier predicts whether an SMS is `ham`, `spam`, or `smishing`. RAG complements that classifier by retrieving relevant evidence from trusted Lebanese cybersecurity sources so that suspicious messages can be explained with traceable guidance.

The notebook follows this progression:

1. collect trusted Lebanese cybersecurity sources,
2. clean and chunk the source documents,
3. create multilingual semantic embeddings,
4. store and retrieve evidence with Chroma,
5. evaluate baseline and metadata-enriched retrieval,
6. test a small instruction-tuned generator under strict grounding rules,
7. add deterministic validation and an extractive fallback,
8. integrate retrieval with the trained Transformer classifier and safety-routing logic.

The generator is treated as an experiment rather than an authority. The final deployed AmanLeb prototype prioritizes deterministic retrieval and extractive evidence when generated text cannot be trusted.

## 1. Setup

In [ ]:
!pip install -q sentence-transformers chromadb beautifulsoup4 requests

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding model loaded.")
print("Embedding dimension:",
      embedding_model.get_sentence_embedding_dimension())

## 2. Trusted Lebanese Cybersecurity Sources

The initial knowledge base uses official Lebanese sources that provide cybersecurity, fraud-prevention, or account-safety guidance. Source metadata is preserved so that every retrieved chunk can be traced back to its organization and URL.

In [ ]:
trusted_sources = [
    {
        "source_id": "isf_sms_fraud_2026",
        "title": "ISF Warning — Fraudulent SMS Impersonating a Money Transfer Company",
        "url": "https://isf.gov.lb/news/fraudulent-text-messages-impersonating-a-money-transfer-company-beware-of-the-theft-of-your-personal-data-and-money/",
        "organization": "Lebanese Internal Security Forces"
    },
    {
        "source_id": "isf_security_awareness",
        "title": "ISF Internet Security Awareness",
        "url": "https://isf.gov.lb/internet-security-awareness/",
        "organization": "Lebanese Internal Security Forces"
    },
    {
        "source_id": "alfa_security_tips",
        "title": "Alfa Security Tips",
        "url": "https://www.alfa.com.lb/en/support/security-tips",
        "organization": "Alfa"
    }
]

pd.DataFrame(trusted_sources)

## 3. Web Extraction

The source webpages are downloaded and converted to plain text. Navigation, scripts, styles, headers, footers, and other non-content elements are removed before further processing.

In [ ]:
import requests
from bs4 import BeautifulSoup

def extract_webpage_text(url):
    response = requests.get(
        url,
        timeout=30,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove elements that usually do not contain useful article content
    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "noscript"
    ]):
        tag.decompose()

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    return " ".join(text.split())

In [ ]:
documents = []

for source in trusted_sources:
    text = extract_webpage_text(source["url"])

    documents.append({
        **source,
        "text": text
    })

    print(
        source["source_id"],
        "→",
        len(text),
        "characters"
    )

### 3.1 Inspect Raw Extraction

A short preview is used during development to confirm that the webpage content was retrieved successfully before embedding.

In [ ]:
for doc in documents:
    print("\n" + "=" * 80)
    print(doc["title"])
    print("=" * 80)
    print(doc["text"][:1500])

## 3. Document Cleaning

The initial web extraction successfully retrieved the source pages, but it also included navigation menus, footer content, and unrelated webpage elements.

Before chunking and embedding the documents, source-specific cleaning is applied so that the RAG knowledge base contains only relevant cybersecurity and scam-prevention information.

In [ ]:
def extract_between(text, start_marker, end_marker=None):
    """
    Keep only the text between two markers.
    Matching is case-insensitive.
    """

    text_lower = text.lower()

    start_index = text_lower.find(start_marker.lower())

    if start_index == -1:
        print(f"WARNING: start marker not found: {start_marker}")
        return text

    cleaned = text[start_index:]

    if end_marker is not None:
        cleaned_lower = cleaned.lower()
        end_index = cleaned_lower.find(end_marker.lower())

        if end_index != -1:
            cleaned = cleaned[:end_index]

    return " ".join(cleaned.split())

### 3.2 Apply Source-Specific Cleaning

Different webpages have different layouts, so source-specific start and end markers are used to keep the relevant cybersecurity content while discarding unrelated page sections.

In [ ]:
cleaned_documents = []

for doc in documents:

    text = doc["text"]

    if doc["source_id"] == "isf_sms_fraud_2026":

        clean_text = extract_between(
            text,
            start_marker="Issued by the General Directorate",
            end_marker="Other Articles"
        )

    elif doc["source_id"] == "isf_security_awareness":

        clean_text = extract_between(
            text,
            start_marker="Your Behaviour on the Internet"
        )

    elif doc["source_id"] == "alfa_security_tips":

        clean_text = extract_between(
            text,
            start_marker="General tips",
            end_marker="ABOUT ALFA"
        )

    else:
        clean_text = text

    cleaned_documents.append({
        **doc,
        "text": clean_text
    })

In [ ]:
for doc in cleaned_documents:

    print("\n" + "=" * 80)
    print(doc["title"])
    print("=" * 80)

    print("Characters:", len(doc["text"]))
    print()

    print(doc["text"][:1500])

## 4. Document Chunking

The cleaned cybersecurity documents are too large to embed and retrieve as single units.

Each document is therefore divided into smaller overlapping chunks. Smaller chunks allow the retrieval system to return the specific part of a source that is relevant to a user's SMS instead of retrieving an entire webpage.

A small overlap is kept between neighboring chunks so that information located near a chunk boundary is not separated completely.

Each chunk also preserves metadata such as its source, organization, URL, and chunk number so that retrieved evidence can later be traced back to its original source.

In [ ]:
print(
    "Embedding model maximum sequence length:",
    embedding_model.max_seq_length
)

Embedding model maximum sequence length: 128


### 4.1 Chunking Function

In [ ]:
def chunk_text(text, chunk_size=80, overlap=20):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        # Stop once this chunk reaches the end of the document
        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

### 4.2 Create Metadata-Preserving Chunks

Each chunk keeps its source ID, title, organization, URL, and chunk index. This metadata is later used for source attribution and same-source neighboring-chunk recovery.

In [ ]:
chunks = []

for doc in cleaned_documents:

    document_chunks = chunk_text(
        doc["text"],
        chunk_size=80,
        overlap=20
    )

    for chunk_index, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f"{doc['source_id']}_{chunk_index}",
            "source_id": doc["source_id"],
            "title": doc["title"],
            "organization": doc["organization"],
            "url": doc["url"],
            "chunk_index": chunk_index,
            "text": chunk
        })

### 4.3 Chunk Statistics

In [ ]:
from collections import Counter

print("Total chunks:", len(chunks))

chunk_counts = Counter(
    chunk["source_id"]
    for chunk in chunks
)

print(chunk_counts)


Total chunks: 67
Counter({'isf_security_awareness': 52, 'alfa_security_tips': 12, 'isf_sms_fraud_2026': 3})


## 5. Semantic Embeddings

Each cleaned document chunk is converted into a dense numerical embedding using the multilingual Sentence Transformer model.

Unlike keyword-based representations, semantic embeddings attempt to represent the meaning of a text in a fixed-dimensional vector space. Chunks with similar meanings should therefore have vectors located closer together even when they do not contain exactly the same words.

The embeddings will later be stored in a Chroma vector database and compared against embeddings generated from suspicious SMS messages.

In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of chunks:", len(chunks))
print("Embedding matrix shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Number of chunks: 67
Embedding matrix shape: (67, 384)


## 6. Baseline Chroma Vector Store

The normalized chunk embeddings are stored in a persistent Chroma collection together with the original chunk text and source metadata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import chromadb
import os

CHROMA_PATH = "/content/drive/MyDrive/AmanLeb/chroma_db"

os.makedirs(CHROMA_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [ ]:
collection = chroma_client.get_or_create_collection(
    name="amanleb_cybersecurity"
)

In [ ]:
ids = [
    chunk["chunk_id"]
    for chunk in chunks
]

documents_for_chroma = [
    chunk["text"]
    for chunk in chunks
]

metadatas = [
    {
        "source_id": chunk["source_id"],
        "title": chunk["title"],
        "organization": chunk["organization"],
        "url": chunk["url"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in chunks
]

In [ ]:
collection.upsert(
    ids=ids,
    documents=documents_for_chroma,
    embeddings=chunk_embeddings.tolist(),
    metadatas=metadatas
)

In [ ]:
print("Chunks in Chroma:", collection.count())

Chunks in Chroma: 67


## 7. Baseline Semantic Retrieval

A synthetic money-transfer phishing SMS is embedded in the same semantic space as the knowledge-base chunks. Chroma then returns the nearest chunks.

In [ ]:
test_sms = """
Your money transfer account has been suspended.
Click the link below and enter your banking information
to restore your account and avoid losing access.
"""

In [ ]:
query_embedding = embedding_model.encode(
    [test_sms],
    normalize_embeddings=True
)

In [ ]:
print(query_embedding.shape)

(1, 384)


In [ ]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

In [ ]:
for i in range(3):

    print("=" * 80)
    print(f"RESULT {i + 1}")
    print()

    print(
        "Source:",
        results["metadatas"][0][i]["title"]
    )

    print(
        "Organization:",
        results["metadatas"][0][i]["organization"]
    )

    print(
        "Distance:",
        results["distances"][0][i]
    )

    print()

    print(results["documents"][0][i])

    print()

RESULT 1

Source: ISF Internet Security Awareness
Organization: Lebanese Internal Security Forces
Distance: 0.9045228958129883

to your bank account through links contained in emails or anonymous text messages. This is typically a phishing campaign used to steal your personal information or bank funds. Always logout of your internet banking accounts immediately when you finish using them. Be cautious, when shopping online, from providing additional information than is not necessary to complete the transaction. Read the experience reviews and opinions, positive or negative, expressed by customers who have used the Internet shopping websites you are planning

RESULT 2

Source: ISF Internet Security Awareness
Organization: Lebanese Internal Security Forces
Distance: 0.9051500558853149

card’s account, in preparation for doing what is necessary to recover any unauthorized transactions. Update the Mobile Wallet on your phone and all the shopping accounts on the Internet where you have used 

## 8. Metadata-Enriched Retrieval

Initial semantic retrieval successfully returned cybersecurity guidance related to phishing and banking-information theft. However, the most specific ISF fraudulent-SMS advisory did not appear among the top retrieved chunks.

One limitation is that document titles and source information were stored only as metadata and were not included in the text used to generate embeddings.

To provide the embedding model with additional semantic context, each chunk is enriched with its source title and organization before embedding. The original chunk text remains unchanged for display and generation.

In [ ]:
embedding_texts = [
    (
        f"Title: {chunk['title']}\n"
        f"Organization: {chunk['organization']}\n"
        f"Content: {chunk['text']}"
    )
    for chunk in chunks
]

A representative enriched embedding string is shown below during development. The original chunk text is still stored separately for display and grounding.

In [ ]:
print(embedding_texts[0])

In [ ]:
chunk_embeddings_v2 = embedding_model.encode(
    embedding_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(chunk_embeddings_v2.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(67, 384)


In [ ]:
collection_v2 = chroma_client.get_or_create_collection(
    name="amanleb_cybersecurity_v2",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

In [ ]:
collection_v2.upsert(
    ids=ids,
    documents=documents_for_chroma,
    embeddings=chunk_embeddings_v2.tolist(),
    metadatas=metadatas
)

print("Chunks stored:", collection_v2.count())

Chunks stored: 67


In [ ]:
query_embedding = embedding_model.encode(
    [test_sms],
    normalize_embeddings=True
)

results_v2 = collection_v2.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

In [ ]:
for i in range(3):

    print("=" * 80)
    print(f"RESULT {i + 1}")

    metadata = results_v2["metadatas"][0][i]

    print("Source:", metadata["title"])
    print("Organization:", metadata["organization"])
    print(
        "Cosine distance:",
        round(results_v2["distances"][0][i], 4)
    )

    print()
    print(results_v2["documents"][0][i])
    print()

## 9. Retrieval API and Multilingual Tests

The final retrieval helper uses the metadata-enriched collection. Several representative queries are defined to test banking phishing, money-transfer scams, SIM-security concerns, Arabic phishing, and a normal message.

The large development printout is intentionally cleared from the submission notebook; the code remains reproducible and can be rerun when needed.

In [ ]:
def retrieve_evidence(query, n_results=3):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    results = collection_v2.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for i in range(n_results):

        metadata = results["metadatas"][0][i]

        retrieved.append({
            "rank": i + 1,
            "source": metadata["title"],
            "organization": metadata["organization"],
            "url": metadata["url"],
            "source_id": metadata["source_id"],
            "chunk_index": metadata["chunk_index"],
            "distance": results["distances"][0][i],
            "text": results["documents"][0][i]
        })

    return retrieved


In [ ]:
def display_retrieval(query, n_results=3):

    print("QUERY:")
    print(query)

    print("\n" + "=" * 80)

    results = retrieve_evidence(
        query,
        n_results=n_results
    )

    for result in results:

        print(f"\nRESULT {result['rank']}")
        print("Source:", result["source"])
        print("Organization:", result["organization"])
        print(
            "Cosine distance:",
            round(result["distance"], 4)
        )

        print()
        print(result["text"])
        print("\n" + "-" * 80)

In [ ]:
retrieval_tests = {

    "money_transfer_phishing": """
    Your money transfer account has been suspended.
    Click this link and enter your banking information
    to restore your account.
    """,

    "bank_phishing": """
    Your bank account has been blocked due to suspicious activity.
    Verify your card number and password using the following link.
    """,

    "sim_security": """
    My SIM card was stolen and I am worried that someone
    could access my accounts or use my mobile number.
    """,

    "arabic_phishing": """
    تم توقيف حسابك. اضغط على الرابط التالي وأدخل معلوماتك
    المصرفية لتفعيل الحساب من جديد.
    """,

    "normal_message": """
    Hey, are we still meeting at university tomorrow at 10?
    """
}

In [ ]:
for test_name, query in retrieval_tests.items():

    print("\n\n")
    print("#" * 100)
    print(test_name.upper())
    print("#" * 100)

    display_retrieval(
        query,
        n_results=3
    )

## 10. Experimental Grounded Response Generation

After validating retrieval, a small instruction-tuned language model (`Qwen/Qwen2.5-1.5B-Instruct`) is tested as an optional response generator.

The generator does **not** replace the classifier or retriever. Its only role is to turn retrieved evidence into a short explanation and recommended actions. Because generated language can introduce unsupported claims or ignore formatting constraints, its output is treated as untrusted until it passes deterministic validation.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

generator_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype="auto",
    device_map="auto"
)

print("Generator loaded.")
print("Device:", generator_model.device)

### 10.1 Grounding and Validation Design

The experimental generator is deliberately given limited authority:

- the Transformer classifier owns the model prediction,
- Chroma retrieves the trusted evidence,
- Qwen receives only the SMS and retrieved evidence,
- Python adds the real source list,
- a deterministic validator checks required sections, bullet structure, evidence citations, and lexical traceability,
- Qwen receives **one** generation attempt,
- if that attempt fails validation, the generated explanation is rejected and AmanLeb uses an extractive fallback built directly from retrieved official evidence.

There is no repair-generation loop in the final notebook design.

In [ ]:
def build_rag_context(retrieved_results):
    """Format retrieved chunks as numbered evidence blocks."""

    context_parts = []

    for i, result in enumerate(retrieved_results, start=1):
        context_parts.append(
            f"""[EVIDENCE {i}]
Source: {result['source']}
Organization: {result['organization']}
Content:
{result['text']}"""
        )

    return "\n\n".join(context_parts)


In [ ]:
STRICT_SYSTEM_PROMPT = """
You are AmanLeb.

Your job is to produce a SHORT evidence-grounded explanation of a suspicious SMS.

IMPORTANT:
- Use ONLY the supplied EVIDENCE.
- Stay very close to the wording of the evidence.
- Prefer extracting or lightly paraphrasing the evidence instead of inventing new wording.
- Do NOT add outside knowledge, assumptions, or likely consequences.
- Do NOT mention malware, ransomware, identity theft, phone-number reputation,
  urgency tactics, or any other risk unless those ideas are explicitly written in the evidence.
- Do NOT classify the SMS.
- Do NOT generate sources.
- Every bullet must end with exactly one citation such as [EVIDENCE 1].
- Write at least one bullet under each required section.
- Keep each bullet concise.

Use exactly these two section headings:

Why this message may be suspicious:
Recommended actions:

Use hyphen bullets only.
"""


In [ ]:
import re
import string


def _normalize_heading(line):
    """Remove common Markdown heading/bold syntax before comparison."""
    line = line.strip()
    line = re.sub(r"^#+\s*", "", line)
    line = line.replace("**", "").replace("__", "")
    return line.strip()


def _content_words(text):
    """
    Tokenize to simple lowercase content words.
    A small stopword list is used only for lexical-grounding validation.
    """
    stopwords = {
        "the", "a", "an", "and", "or", "to", "of", "in", "on", "for",
        "is", "are", "was", "were", "be", "been", "being", "this", "that",
        "these", "those", "it", "its", "with", "as", "by", "from", "at",
        "if", "you", "your", "they", "their", "them", "any", "can", "could",
        "should", "would", "may", "might", "will", "do", "not"
    }

    words = re.findall(r"[A-Za-z0-9']+", text.lower())
    return {w for w in words if len(w) > 2 and w not in stopwords}


def _lexical_grounding_score(bullet_text, evidence_text):
    """
    Fraction of meaningful bullet words that also appear in the cited evidence.
    This is a conservative heuristic for evidence traceability, not semantic proof.
    """
    bullet_words = _content_words(bullet_text)
    evidence_words = _content_words(evidence_text)

    if not bullet_words:
        return 0.0

    return len(bullet_words & evidence_words) / len(bullet_words)


def validate_grounded_response(response, retrieved_results, min_overlap=0.55):
    """
    Validate:
    1. required sections,
    2. bullet format,
    3. valid evidence citations,
    4. obvious placeholder text,
    5. lexical grounding against the cited evidence.

    This is intentionally conservative.
    """

    problems = []

    suspicious_header = "Why this message may be suspicious:"
    actions_header = "Recommended actions:"

    forbidden_placeholders = [
        "evidence-supported statement",
        "evidence-supported action",
        "statement supported by evidence",
        "action supported by evidence",
    ]

    response_lower = response.lower()

    for phrase in forbidden_placeholders:
        if phrase in response_lower:
            problems.append(
                f"Placeholder/template text was copied: '{phrase}'"
            )

    # Normalize non-empty lines so Markdown bold/headings do not cause false failures.
    raw_lines = [
        line.strip()
        for line in response.splitlines()
        if line.strip()
    ]

    normalized_lines = [_normalize_heading(line) for line in raw_lines]

    if suspicious_header not in normalized_lines:
        problems.append(f"Missing section: {suspicious_header}")

    if actions_header not in normalized_lines:
        problems.append(f"Missing section: {actions_header}")

    current_section = None
    suspicious_bullets = 0
    action_bullets = 0

    for raw_line, norm_line in zip(raw_lines, normalized_lines):

        if norm_line == suspicious_header:
            current_section = "suspicious"
            continue

        if norm_line == actions_header:
            current_section = "actions"
            continue

        # Accept only hyphen bullets so the output format remains deterministic.
        if raw_line.startswith("-"):

            if current_section is None:
                problems.append(
                    f"Bullet appears outside a required section: {raw_line}"
                )
                continue

            citations = re.findall(r"\[EVIDENCE (\d+)\]", raw_line)

            if len(citations) != 1:
                problems.append(
                    f"Each bullet must contain exactly one evidence citation: {raw_line}"
                )
                continue

            evidence_number = int(citations[0])

            if not 1 <= evidence_number <= len(retrieved_results):
                problems.append(
                    f"Invalid evidence citation [EVIDENCE {evidence_number}]: {raw_line}"
                )
                continue

            if not re.search(r"\[EVIDENCE \d+\]\s*$", raw_line):
                problems.append(
                    f"Citation must be at the end of the bullet: {raw_line}"
                )

            # Remove bullet marker and citation before checking lexical grounding.
            bullet_text = raw_line[1:].strip()
            bullet_text = re.sub(
                r"\s*\[EVIDENCE \d+\]\s*$",
                "",
                bullet_text
            )

            if len(bullet_text.split()) < 5:
                problems.append(
                    f"Bullet is too short/generic: {raw_line}"
                )

            cited_evidence = retrieved_results[evidence_number - 1]["text"]

            grounding_score = _lexical_grounding_score(
                bullet_text,
                cited_evidence
            )

            if grounding_score < min_overlap:
                problems.append(
                    f"Low lexical grounding ({grounding_score:.2f}) "
                    f"for [EVIDENCE {evidence_number}]: {raw_line}"
                )

            if current_section == "suspicious":
                suspicious_bullets += 1
            elif current_section == "actions":
                action_bullets += 1

        else:
            problems.append(
                f"Unexpected text outside the required hyphen-bullet format: {raw_line}"
            )

    if suspicious_bullets == 0:
        problems.append(
            "No grounded bullet under the suspicious-message section."
        )

    if action_bullets == 0:
        problems.append(
            "No grounded bullet under the recommended-actions section."
        )

    return len(problems) == 0, problems


In [ ]:
def expand_with_same_source_neighbor(retrieved_results):
    """
    Add the immediately following chunk from the same official source when it
    exists. This helps complete sentences/actions that were cut by chunking.

    The neighbor is taken from the already-cleaned in-memory `chunks` list,
    not from a different semantically similar source.
    """

    expanded = list(retrieved_results)

    if not retrieved_results:
        return expanded

    top = retrieved_results[0]

    next_index = top["chunk_index"] + 1

    for chunk in chunks:

        if (
            chunk["source_id"] == top["source_id"]
            and chunk["chunk_index"] == next_index
        ):
            expanded.append({
                "rank": len(expanded) + 1,
                "source": chunk["title"],
                "organization": chunk["organization"],
                "url": chunk["url"],
                "source_id": chunk["source_id"],
                "chunk_index": chunk["chunk_index"],
                "distance": None,
                "text": chunk["text"]
            })
            break

    return expanded


def split_evidence_sentences(text):
    """Simple sentence splitter after normalizing whitespace."""
    text = " ".join(text.split())

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def extract_action_phrases(text):

    text = " ".join(text.split())

    action_starters = [
        "Do not click",
        "Do not enter",
        "Do not forward",
        "Verify "
    ]

    positions = []

    for starter in action_starters:

        start = text.find(starter)

        if start != -1:
            positions.append(
                (start, starter)
            )

    positions.sort()

    actions = []

    for i, (start, starter) in enumerate(positions):

        if i + 1 < len(positions):
            end = positions[i + 1][0]
        else:
            end = len(text)

        action = text[start:end].strip()

        action = action.strip(" ;")

        if action:
            actions.append(action)

    return actions


def build_extractive_fallback(retrieved_results):
    """
    Build a conservative answer directly from official retrieved evidence.

    The fallback:
    - may use the next chunk from the SAME source to complete cut text,
    - selects descriptive fraud/risk evidence separately from actions,
    - keeps the original source wording,
    - never invents a new factual claim.
    """

    evidence = expand_with_same_source_neighbor(
        retrieved_results
    )

    suspicious_keywords = {
        "fraud", "fraudulent", "scam", "phishing",
        "steal", "theft", "personal", "banking",
        "financial", "suspicious", "link", "messages"
    }

    action_prefixes = (
        "do not ",
        "verify ",
        "report ",
        "contact ",
        "avoid ",
        "never ",
        "always ",
        "stop ",
        "change ",
        "lock ",
        "update "
    )

    suspicious_candidates = []
    action_candidates = []

    for evidence_number, result in enumerate(evidence, start=1):

        # -------------------------
        # Descriptive evidence
        # -------------------------
        for sentence in split_evidence_sentences(result["text"]):

            sentence_lower = sentence.lower().strip()

            # Never use imperative advice as the "why suspicious" explanation.
            if sentence_lower.startswith(action_prefixes):
                continue

            # Also skip bridge text that merely introduces a list of actions.
            if (
                "urges the public" in sentence_lower
                or sentence_lower.endswith("to:")
            ):
                continue

            keyword_hits = sum(
                1
                for keyword in suspicious_keywords
                if keyword in sentence_lower
            )

            if keyword_hits >= 2:
                suspicious_candidates.append(
                    (keyword_hits, evidence_number, sentence)
                )

        # -------------------------
        # Action evidence
        # -------------------------
        for action in extract_action_phrases(result["text"]):
            action_candidates.append(
                (evidence_number, action)
            )

    # Strongest descriptive sentence first.
    suspicious_candidates.sort(
        key=lambda item: item[0],
        reverse=True
    )

    selected_suspicious = []
    seen_suspicious = set()

    for _, evidence_number, sentence in suspicious_candidates:

        key = sentence.lower()

        if key not in seen_suspicious:
            selected_suspicious.append(
                (evidence_number, sentence)
            )
            seen_suspicious.add(key)

        # One strong explanation is enough for the fallback.
        if len(selected_suspicious) == 1:
            break

    selected_actions = []
    seen_actions = set()

    for evidence_number, action in action_candidates:

        # Remove obvious trailing conjunction/fragment artifacts.
        action = action.strip(" ;")

        # Do not keep an obviously incomplete final fragment.
        if action.lower().endswith(
            ("with the relevant", "through the relevant", "directly with the relevant")
        ):
            continue

        key = action.lower()

        if key not in seen_actions:
            selected_actions.append(
                (evidence_number, action)
            )
            seen_actions.add(key)

        if len(selected_actions) == 3:
            break

    lines = [
        "Why this message may be suspicious:"
    ]

    if selected_suspicious:
        for evidence_number, sentence in selected_suspicious:
            lines.append(
                f"- {sentence} [EVIDENCE {evidence_number}]"
            )
    else:
        lines.append(
            "- The retrieved official evidence was insufficient "
            "to produce a specific descriptive explanation."
        )

    lines.append("")
    lines.append("Recommended actions:")

    if selected_actions:
        for evidence_number, action in selected_actions:
            lines.append(
                f"- {action} [EVIDENCE {evidence_number}]"
            )
    else:
        lines.append(
            "- No complete action statement could be extracted "
            "from the retrieved official evidence."
        )

    return "\n".join(lines)


In [ ]:
def generate_candidate(messages):
    """Generate one deterministic candidate answer from Qwen."""

    text = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = generator_tokenizer(
        [text],
        return_tensors="pt"
    ).to(generator_model.device)

    generated_ids = generator_model.generate(
        **model_inputs,
        max_new_tokens=220,
        do_sample=False
    )

    # Keep only the newly generated tokens.
    generated_ids = generated_ids[
        :,
        model_inputs.input_ids.shape[1]:
    ]

    response = generator_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response.strip()


In [ ]:
def generate_grounded_explanation(
    sms,
    n_results=1
):
    """
    Hybrid safe generation:
    retrieve -> one LLM attempt -> validate -> extractive fallback if needed.
    """

    retrieved = retrieve_evidence(
        sms,
        n_results=n_results
    )

    context = build_rag_context(retrieved)

    messages = [
        {
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""SMS MESSAGE:
{sms}

TRUSTED EVIDENCE:
{context}

Write a short answer using ONLY the trusted evidence.

Use exactly these headings:
Why this message may be suspicious:
Recommended actions:

Use hyphen bullets only.
Every bullet must end with its [EVIDENCE #] citation.
Stay very close to the wording in the evidence."""
        }
    ]

    response = generate_candidate(messages)

    valid, problems = validate_grounded_response(
        response,
        retrieved_results=retrieved
    )

    print(
        "LLM generation:",
        "VALID" if valid else "INVALID"
    )

    if valid:
        return response, retrieved, "llm", problems

    print(
        "Using safe extractive fallback from retrieved evidence."
    )

    fallback = build_extractive_fallback(
        retrieved
    )

    return fallback, retrieved, "extractive_fallback", problems


In [ ]:
def display_amanleb_result(
    sms,
    predicted_label,
    n_results=1
):
    """
    Display the final AmanLeb result.
    """

    response, retrieved, generation_mode, problems = (
        generate_grounded_explanation(
            sms=sms,
            n_results=n_results
        )
    )

    print("\nAnalysis status:", predicted_label)
    print("Explanation mode:", generation_mode)
    print()

    print(response)

    print("\nSources:")

    seen_sources = set()

    for result in retrieved:
        source_key = (
            result["source"],
            result["organization"],
            result["url"]
        )

        if source_key not in seen_sources:
            print(f"- {result['source']}")
            print(
                f"  Organization: "
                f"{result['organization']}"
            )
            print(
                f"  URL: "
                f"{result['url']}"
            )
            seen_sources.add(source_key)


## 11. Transformer Classifier Integration and Safety Routing

The saved weighted multilingual DistilBERT classifier is loaded and connected to the retrieval system.

The classifier returns the three class probabilities. AmanLeb then applies a prototype safety-routing rule:

- a high-confidence `ham` prediction (at least 0.80) is treated as likely safe and does not require retrieval,
- a lower-confidence `ham` prediction is marked **Needs Review**,
- `spam` or `smishing` predictions are marked **Suspicious**.

The 0.80 threshold is a conservative prototype heuristic rather than an experimentally optimized threshold.

In [ ]:
test_sms = """
Your money transfer account has been suspended.
Click the link below and enter your banking information
to restore your account.
"""


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

CLASSIFIER_PATH = "/content/drive/MyDrive/AmanLeb/amanleb_final_transformer"

classifier_tokenizer = AutoTokenizer.from_pretrained(
    CLASSIFIER_PATH
)

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_PATH
)

classifier_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

classifier_model.to(device)

print("Classifier loaded on:", device)

In [ ]:
import torch.nn.functional as F


def predict_sms_class(sms):

    inputs = classifier_tokenizer(
        sms,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():

        outputs = classifier_model(
            **inputs
        )

    probabilities = F.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_id = torch.argmax(
        probabilities
    ).item()

    predicted_label = classifier_model.config.id2label[
        predicted_id
    ]

    probs = {
        classifier_model.config.id2label[i]:
        probabilities[i].item()
        for i in range(len(probabilities))
    }

    return predicted_label, probs

In [ ]:
def amanleb_pipeline(sms):

    predicted_label, probs = predict_sms_class(sms)

    ham_probability = probs["ham"]

    suspicious_probability = (
        probs["spam"] +
        probs["smishing"]
    )

    # Show classifier probabilities
    print(
        f"Model probabilities → "
        f"ham: {probs['ham']:.3f}, "
        f"spam: {probs['spam']:.3f}, "
        f"smishing: {probs['smishing']:.3f}"
    )

    # --------------------------------------------------
    # CASE 1: High-confidence HAM
    # --------------------------------------------------

    if (
        predicted_label == "ham"
        and ham_probability >= 0.80
    ):

        print("\nModel prediction: ham")
        print("Safety status: likely safe")

        print(
            "This message was classified as ham "
            "with high confidence."
        )

        return

    # --------------------------------------------------
    # CASE 2: Low-confidence HAM
    # --------------------------------------------------

    if predicted_label == "ham":

        print("\nModel prediction: ham")
        print("Safety status: needs review")

        print(
            f"Combined suspicious probability: "
            f"{suspicious_probability:.3f}"
        )

        # Low-confidence HAM is sent to RAG
        # for additional safety analysis.
        display_amanleb_result(
            sms=sms,
            predicted_label="needs review",
            n_results=1
        )

        return

    # --------------------------------------------------
    # CASE 3: SPAM or SMISHING
    # --------------------------------------------------

    print(
        f"\nModel prediction: {predicted_label}"
    )

    print("Safety status: suspicious")

    display_amanleb_result(
        sms=sms,
        predicted_label=predicted_label,
        n_results=1
    )

### 11.1 End-to-End Notebook Test

The following test exercises the classifier, safety routing, retrieval, validation, and fallback behavior together. The saved development output is cleared so that the notebook does not preserve an outdated four-action fallback result; rerunning the cell uses the current three-action fallback.

In [ ]:
amanleb_pipeline(test_sms)

## 12. Final RAG and Deployment Decision

The experiments showed that multilingual semantic retrieval can retrieve relevant Lebanese cybersecurity evidence, including cross-lingual retrieval for Arabic queries.

The small Qwen generator produced fluent responses but was not consistently reliable under strict grounding and formatting constraints. It sometimes introduced unsupported claims or failed to follow the required evidence-citation format.

AmanLeb therefore adopts the following safety principle:

1. retrieve evidence from trusted official sources,
2. allow generated text only when it passes deterministic validation,
3. otherwise fall back to extractive evidence from the retrieved source,
4. keep the classifier prediction and source attribution deterministic.

For the **final Streamlit prototype**, Qwen is not required at runtime. The deployed application uses the trained Transformer classifier, confidence-based safety routing, multilingual semantic retrieval, and deterministic extractive guidance from trusted Lebanese sources. This reduces local resource requirements and prioritizes traceability over fluency.

### Current Limitation

The knowledge base is intentionally small. Retrieval can therefore return a broadly relevant source without always finding a sufficiently specific sentence or action for every scam type. Expanding the trusted Lebanese source collection and improving evidence extraction are natural future improvements.